# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and accessible via:
[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access metadata object (do not index or iterate over it)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, fields, and their `@id` values.

First, list the available record sets in the dataset with their `@id`. Then for each, list the available fields and their `@id`.

In [ ]:
# Show all record sets and their fields

record_sets = dataset.record_sets()
print("Record Sets in the dataset:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name']} (type: {rs['@type']})")
    print("  Fields:")
    for field in rs['fields']:
        print(f"    - {field['@id']}: {field['name']} (type: {field['dataType']})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract the data from the primary survey record set (for demonstration, replace with appropriate `@id` as needed based on the overview above):

In [ ]:
# Example: Extract data from all record sets
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Display the columns for the first record set
first_rs_id = record_set_ids[0] if record_set_ids else None
if first_rs_id is not None:
    print(f"Record set {first_rs_id} columns:", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will select a numeric field (e.g., GAD-7 score) referenced by its field `@id` and apply basic transformations. Adjust field IDs based on available columns and the record set overview above.

In [ ]:
# Example EDA with a numeric field (replace with correct @id as explored above)
# Identify a suitable numeric field from a record set, e.g. 'gad7_score', assuming its @id is 'cr:gad7_score'
record_set_id = first_rs_id
numeric_field_id = None
for rs in dataset.record_sets():
    if rs['@id'] == record_set_id:
        # Find first field with numeric data
        for field in rs['fields']:
            if 'Float' in field['dataType'] or 'Integer' in field['dataType']:
                numeric_field_id = field['@id']
                break
        break
# For demonstration, set threshold for filtering
threshold = 10
if numeric_field_id and numeric_field_id in dataframes[record_set_id].columns:
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())
    
    # Group by a categorical field (e.g., gender, if present)
    group_field = None
    for field in dataset.record_sets()[0]['fields']:
        if 'gender' in field['name'].lower():
            group_field = field['@id']
            break
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example visualization: distribution of numeric field
if numeric_field_id and numeric_field_id in dataframes[record_set_id].columns:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[record_set_id][numeric_field_id].dropna(), bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot by group if applicable
if group_field and group_field in dataframes[record_set_id].columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=dataframes[record_set_id][group_field], y=dataframes[record_set_id][numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, inspect, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.
- All dataset entities (record sets, fields) were referenced using their `@id`.
- Data was loaded and overviewed; numeric and categorical fields were selected dynamically based on schema.
- Exploratory and visualization steps included filtering, normalization, grouping, and plotting.

Further analysis could include missing data handling, cross-field correlation, or predictive modeling on survey responses.